In [2]:
import torch
import torch.nn as nn
import math

Positional Encoding

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Add batch dimension
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x is assumed to be (batch_size, seq_len, d_model)
        # Add positional encoding to the input embeddings
        x = x + self.pe[:, :x.size(1)]
        return x

 Multi-Head Self-Attention

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)
        self.out_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1. Linear projections for Q, K, V
        Q = self.q_linear(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. Scaled Dot-Product Attention
        # (batch_size, num_heads, seq_len, head_dim) @ (batch_size, num_heads, head_dim, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 3. Apply mask (e.g., for padding)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # 4. Softmax to get attention weights
        attention_weights = torch.softmax(scores, dim=-1)

        # 5. Multiply by Value to get the weighted sum
        # (batch_size, num_heads, seq_len, seq_len) @ (batch_size, num_heads, seq_len, head_dim)
        # -> (batch_size, num_heads, seq_len, head_dim)
        attended_output = torch.matmul(attention_weights, V)

        # 6. Concatenate heads and apply final linear layer
        # (batch_size, seq_len, num_heads * head_dim) = (batch_size, seq_len, embed_dim)
        attended_output = attended_output.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.head_dim)
        output = self.out_linear(attended_output)
        return output

Position-wise Feed-Forward Network

In [5]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim):
        super().__init__()
        self.linear1 = nn.Linear(embed_dim, ff_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(ff_dim, embed_dim)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

Encoder Layer (combining components)

In [6]:
class EncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate):
        super().__init__()
        self.self_attn = MultiHeadAttention(embed_dim, num_heads)
        self.feed_forward = PositionwiseFeedForward(embed_dim, ff_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x, mask=None):
        # 1. Multi-Head Self-Attention sub-layer
        attn_output = self.self_attn(x, x, x, mask)
        # Add & Norm for self-attention
        x = self.norm1(x + self.dropout(attn_output))

        # 2. Position-wise Feed-Forward sub-layer
        ff_output = self.feed_forward(x)
        # Add & Norm for feed-forward
        x = self.norm2(x + self.dropout(ff_output))
        return x

Full Encoder (stack of Encoder Layers)

In [7]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_layers, num_heads, ff_dim, dropout_rate, max_len=5000):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = PositionalEncoding(embed_dim, max_len)
        self.layers = nn.ModuleList([EncoderLayer(embed_dim, num_heads, ff_dim, dropout_rate) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, src, src_mask):
        # 1. Token Embedding
        x = self.token_embedding(src)
        # 2. Add Positional Encoding
        x = self.dropout(self.positional_encoding(x))

        # 3. Pass through N encoder layers
        for layer in self.layers:
            x = layer(x, src_mask)
        return x

Example Usage (conceptual)

In [8]:
vocab_size = 10000
embed_dim = 512
num_layers = 6
num_heads = 8
ff_dim = 2048
dropout_rate = 0.1

encoder = Encoder(vocab_size, embed_dim, num_layers, num_heads, ff_dim, dropout_rate)
#
# Example input: a batch of sentences represented as token IDs
# (batch_size, seq_len)
src_tokens = torch.randint(0, vocab_size, (32, 50))

# Example source mask (1 for actual tokens, 0 for padding)
# (batch_size, 1, 1, seq_len) for broadcasting with attention scores
src_mask = (src_tokens != 0).unsqueeze(1).unsqueeze(2)

encoded_representation = encoder(src_tokens, src_mask)
print(encoded_representation.shape) # Expected: (batch_size, seq_len, embed_dim)

torch.Size([32, 50, 512])


In [9]:
encoded_representation

tensor([[[-0.4913, -0.3514, -0.9069,  ...,  0.6265,  0.5924,  0.9014],
         [-0.1799, -1.6871,  0.9713,  ...,  1.1191,  0.8266, -0.0200],
         [ 1.1852, -1.2229,  0.0794,  ..., -0.1273, -0.5572,  0.2010],
         ...,
         [ 0.0428,  0.0643, -0.3157,  ...,  0.6635,  1.2440, -0.4601],
         [-1.0388, -0.3208, -1.1691,  ...,  0.4406,  0.0780,  0.5079],
         [-2.2552, -0.7106, -0.9383,  ..., -1.1100, -0.0890,  1.3529]],

        [[-1.1021, -0.5044, -0.4257,  ..., -0.7547,  0.5273,  0.4966],
         [-0.0570, -0.4492, -1.4491,  ...,  0.7155,  0.1365,  0.3561],
         [ 0.1132, -0.9409,  0.5050,  ...,  0.2949, -0.9865,  1.2855],
         ...,
         [-0.9506, -0.3227,  0.2799,  ...,  0.8650,  1.9212,  2.0442],
         [-0.9188, -0.7511,  0.0735,  ..., -0.8512,  0.2300,  0.1829],
         [-3.2319,  1.5317, -0.3669,  ...,  1.0377,  0.6652,  1.0620]],

        [[-0.4788,  0.3025,  0.2164,  ..., -0.2552,  0.4965,  0.7785],
         [-0.5725,  0.7927,  1.3984,  ...,  0